# 🚀 Fine-tuning Qwen2.5-Coder-3B-Instruct cho Bug Detection

Notebook này chạy toàn bộ training pipeline bao gồm:
1. Kiểm tra GPU
2. Setup môi trường
3. Load config & model
4. Setup LoRA adapter
5. Training với SFTTrainer
6. Lưu kết quả

## 1. Kiểm tra GPU

In [ ]:
# 1. Clone code từ repo GitHub của bạn về Colab
!git clone https://github.com/NguyenQuangAnh1112/finetune-for-detect-bug.git

# 2. Cài đặt các thư viện cần thiết
%cd finetune-for-detect-bug
!pip install -e .
!pip install -r requirements.txt # Nếu bạn có file này, hoặc pip install dataclasses transformers peft datasets trl mlflow

# 3. Chạy lại cell import ở bên dưới sẽ hết lỗi!
import sys
import os
sys.path.insert(0, "/content/finetune-for-detect-bug")
print("✅ Sẵn sàng chạy code!")


In [ ]:
# Cài đặt ép phiên bản chạy ổn định nhất cho QLT/LoRA tuning trên Colab
!pip install -q -U accelerate==0.34.2
!pip install -q -U transformers==4.44.2
!pip install -q -U peft==0.12.0
!pip install -q -U trl==0.9.6
!pip install -q -U datasets==2.21.0
!pip install -q -U mlflow
# Cài đặt thư viện của chính project bạn
!pip install -e .

# Quan trọng: Đôi lúc Colab bắt phải KHỞI ĐỘNG LẠI KERNEL sau khi cập nhật thư viện
# Hãy vào Runtime -> Restart session (hoặc Runtime -> Khởi động lại phiên bản) 
# TRƯỚC khi chạy phần code import `from src.training...`


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 30.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.29.0 requires accelerate>=1.4.0, but you have accelerate 0.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 111.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.5 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.p

In [15]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    # Đã fix lỗi total_mem -> total_memory
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_total:.1f} GB total, {vram_free:.1f} GB free")
else:
    print("⚠️ Không tìm thấy GPU! Training sẽ rất chậm trên CPU.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB total, 14.6 GB free


## 2. Setup môi trường

Thêm thư mục gốc vào `sys.path` để import được các module từ `src/`.

In [16]:
import os
import sys

# Vì notebook nằm ở thư mục gốc, ta chỉ cần add absolute path của current directory
PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Project root added to sys.path: {PROJECT_ROOT}")

✅ Project root added to sys.path: /content/finetune-for-detect-bug


## 3. Load Configuration

In [17]:
from src.utils.config import ConfigurationManager

config_manager = ConfigurationManager()
model_config = config_manager.get_model_config()
training_config = config_manager.get_training_config()

print("=== Model Config ===")
print(f"  Model: {model_config.model_name}")
print(f"  use_4bit: {model_config.use_4bit}")
print("\n=== Training Config ===")
print(f"  Epochs: {training_config.num_train_epochs}")
print(f"  Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

=== Model Config ===
  Model: Qwen/Qwen2.5-Coder-3B-Instruct
  use_4bit: False

=== Training Config ===
  Epochs: 3
  Effective batch size: 8


## 4. Load Model

In [18]:
from src.training.model_trainer import ModelTrainer

trainer_obj = ModelTrainer(config=training_config, model_config=model_config)
print(f"\n✅ Model loaded: {model_config.model_name}")

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

## 5. Setup LoRA Adapter

In [ ]:
trainer_obj.setup_peft_model()
print("\n✅ LoRA adapter applied.")

## 6. Kiểm tra VRAM trước khi train

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM allocated: {allocated:.2f} GB")
    print(f"VRAM reserved:  {reserved:.2f} GB")
    print(f"VRAM total:     {total:.1f} GB")
    print(f"VRAM free:      {total - reserved:.2f} GB")
else:
    print("⚠️ Không có GPU.")

## 7. Training 🔥

Bắt đầu fine-tuning với SFTTrainer + MLflow tracking.

In [ ]:
trainer = trainer_obj.train()
print("\n✅ Training complete!")

## 8. Lưu LoRA Adapter

In [ ]:
save_path = trainer_obj.save(trainer)
print(f"\n✅ Adapter saved at: {save_path}")